# Dim_Hotel

In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, current_timestamp
from pyspark.sql import functions as F

# =============================================================
# CONFIGURATION
# =============================================================
print("="*80)
print("CREATING DIM_HOTEL FROM SILVER LAYER")
print("="*80)

silver_hotels_path = "s3://travel-analytics-bronze/delta/silver/hotels/"
gold_dim_hotel_path = "s3://travel-analytics-bronze/delta/gold/Dim_Hotel/"

# =============================================================
# STEP 1: LOAD SILVER DATA (Already Transformed)
# =============================================================
print("\nSTEP 1: Loading Silver Hotels Data...")

silver_df = spark.read.format("delta").load(silver_hotels_path)

print(f"Loaded {silver_df.count():,} hotel records from silver")

# =============================================================
# STEP 2: ADD SURROGATE KEY & SELECT REQUIRED COLUMNS
# =============================================================
print("\nSTEP 2: Creating Dimension Structure...")

dim_hotel_df = (
    silver_df
    
    # Add surrogate key
    .withColumn("Dim_Hotel_SK", monotonically_increasing_id() + 1)
    
    # Select and rename columns to match dimension schema
    .select(
        col("Dim_Hotel_SK"),                           # PK - Surrogate Key
        col("Hotel_Id").alias("Hotel_ID_BK"),          # Business Key
        col("Hotel_Name"),
        col("Hotel_Address"),
        col("City"),
        col("Country"),
        col("Room_Count"),
        col("Star_Rating"),
        col("Hotel_Score"),
        col("Updated_At")
    )
)

print(f" Dimension structure created with {dim_hotel_df.count():,} records")

CREATING DIM_HOTEL FROM SILVER LAYER

STEP 1: Loading Silver Hotels Data...
Loaded 1,470 hotel records from silver

STEP 2: Creating Dimension Structure...
 Dimension structure created with 1,470 records


In [0]:
dim_hotel_df.display()

Dim_Hotel_SK,Hotel_ID_BK,Hotel_Name,Hotel_Address,City,Country,Room_Count,Star_Rating,Hotel_Score,Updated_At
1,1021,Austria Trend Hotel Astoria Wien,Pontini Maysedergasse Knrntner Viertel Kg Innere Stadt Innere Stadt Vienna 1010 Austria,Vienna,Austria,284,4.5,8.5,2025-12-12T01:04:52.921Z
2,94,Lansbury Heritage Hotel,Lansbury Heritage Hotel 117 Poplar High Street Poplar London Borough Of Tower Hamlets London Greater London England E14 0ae United Kingdom,London,United Kingdom,272,5.0,9.4,2025-12-12T01:04:52.921Z
3,1007,Aparthotel Atenea Barcelona,Hotel Atenea Carrer De Joaquim Molins Les Corts Barcelona Barcelones Barcelona Catalonia 08028 Spain,Barcelona,Spain,233,4.0,8.0,2025-12-12T01:04:52.921Z
4,1257,Arthotel Ana Prime,41 Schnbrunner Strae Nikolsdorf Kg Margareten Margareten Vienna 1050 Austria,Vienna,Austria,257,4.0,8.4,2025-12-12T01:04:52.921Z
5,1329,Hotel Marconi,Centrale Via Martiri Oscuri Greco Milan Lombardy 20172 Italy,Milan,Italy,206,4.0,8.3,2025-12-12T01:04:52.921Z
6,991,Club Hotel Cortina,132 Hietzinger Hauptstrae Kg Ober St. Veit Hietzing Vienna 1130 Austria,Vienna,Austria,202,4.0,8.2,2025-12-12T01:04:52.921Z
7,29,Shangri La Hotel Paris,Shangri La Hotel 10 Avenue Diena Quartier De Chaillot 16th Arrondissement Paris Ile De France Metropolitan France 75116 France,Paris,France,315,5.0,9.3,2025-12-12T01:04:52.921Z
8,569,Washington Mayfair Hotel,35 Park Lane St. Jamess Mayfair City Of Westminster London Greater London England W1k 1pn United Kingdom,London,United Kingdom,169,3.5,7.9,2025-12-12T01:04:52.921Z
9,803,Axel Hotel Barcelona Urban Spa Adults Only,267 Carrer Del Consell De Cent Lantiga Esquerra De Leixample Eixample Barcelona Barcelones Barcelona Catalonia 08011 Spain,Barcelona,Spain,154,4.0,8.0,2025-12-12T01:04:52.921Z
10,362,The One Barcelona Gl,Consolat De Qatar 277 Carrer De Provenca La Dreta De Leixample Eixample Barcelona Barcelones Barcelona Catalonia 08037 Spain,Barcelona,Spain,296,5.0,9.4,2025-12-12T01:04:52.921Z


In [0]:

# =============================================================
# STEP 3: WRITE TO GOLD LAYER
# =============================================================
print("\nSTEP 3: Writing to Gold Layer...")

dim_hotel_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_dim_hotel_path)

print(f"   ✅ Saved to: {gold_dim_hotel_path}")


STEP 3: Writing to Gold Layer...
   ✅ Saved to: s3://travel-analytics-bronze/delta/gold/Dim_Hotel/


In [0]:
# =============================================================
# STEP 4: CREATE MANAGED TABLE
# =============================================================
print("\nSTEP 4: Creating Managed Table...")

# Create gold schema if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS awsdata.gold")

# Create table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS awsdata.gold.Dim_Hotel (
        Dim_Hotel_SK BIGINT COMMENT 'Surrogate key',
        Hotel_ID_BK INTEGER COMMENT 'Business key from source system',
        Hotel_Name STRING,
        Hotel_Address STRING,
        City STRING,
        Country STRING,
        Room_Count INTEGER,
        Star_Rating DOUBLE COMMENT 'Rating from organization',
        Hotel_Score DOUBLE COMMENT 'Score from users',
        Updated_At TIMESTAMP
    )
    USING DELTA
    LOCATION '{gold_dim_hotel_path}'
    COMMENT 'Hotel dimension - SCD Type 1'
""")

print("   ✅ Table created: awsdata.gold.Dim_Hotel")

# =============================================================
# STEP 5: OPTIMIZE
# =============================================================
print("\nSTEP 5: Optimizing...")

spark.sql("OPTIMIZE awsdata.gold.Dim_Hotel")
spark.sql("ANALYZE TABLE awsdata.gold.Dim_Hotel COMPUTE STATISTICS")

print("   ✅ Optimization complete")

# =============================================================
# STEP 6: VALIDATION
# =============================================================
print("\n" + "="*80)
print("VALIDATION")
print("="*80)

result_df = spark.table("awsdata.gold.Dim_Hotel")

print(f"\n✅ Dim_Hotel created successfully!")
print(f"   Records: {result_df.count():,}")
print(f"   Columns: {len(result_df.columns)}")
print(f"   Location: {gold_dim_hotel_path}")

print("\n📄 Sample:")
display(result_df.limit(10))

print("\n" + "="*80)
print("✅ COMPLETE!")
print("="*80)